# PCBSegClassNet — Colab Training

Train PCBSegNet (segmentation) and PCBClassNet (classification) on Google Colab GPU.

**Why Colab?** Local 8 GB GPU (e.g. RTX 4060 Ti) is too tight for `batch=16` at 512×512 input — decoder activation alone is ~4 GB. Colab T4 (16 GB) or A100 (40 GB) handles it comfortably.

## Before you run
1. **Runtime → Change runtime type → GPU** (T4 / A100 / L4 — whatever you have).
2. Have your dataset ready as a zip in Google Drive (see *Data layout* below).
3. Mount Drive when prompted in the relevant cell.

## 1. GPU sanity check

In [ ]:
!nvidia-smi

## 2. Clone the repo

If you forked it, change the URL to your fork.

In [ ]:
%cd /content
!rm -rf PCBSegClassNet
!git clone -b colab https://github.com/ironmanizawesome/PCBSegClassNet.git
%cd PCBSegClassNet

## 3. Set up Python 3.10 + TF 2.10 stack

Colab's default Python is now 3.12, but TF 2.10 only ships wheels for Python 3.7–3.10. Use `condacolab` to swap the runtime to a Python 3.10 base, then pin the TF 2.10 stack on top.

**Why TF 2.10?** This codebase calls `tf.keras.backend.dot` / `backend.transpose` and a couple of other APIs that broke in Keras 3 (TF 2.16+). TF 2.10 is verified to run end-to-end.

> ⚠️ The next cell **restarts the kernel** automatically. Wait for it to reconnect, then continue with the cells after it. The cloned repo at `/content/PCBSegClassNet` survives the restart (it lives on Colab's local disk).

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()  # kernel restarts automatically — rerun cells below afterwards

In [ ]:
!pip install -q \
    tensorflow==2.10.1 \
    keras==2.10.0 \
    tensorflow-estimator==2.10.0 \
    protobuf==3.19.6 \
    numpy==1.24.4

!pip install -q albumentations==1.4.18 opencv-python-headless pyyaml tqdm pandas scikit-learn

In [ ]:
import sys, tensorflow as tf
print("Python:", sys.version.split()[0])
print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 4. Mount Drive and unpack the raw FPIC archive

This notebook does the **entire data prep pipeline** (mask generation + patches + train/val split) in Colab so you only need to upload the raw FPIC images + annotations (~7 GB) instead of the processed dataset (~18 GB).

### Data layout expected on Drive
Zip the **raw** FPIC images + annotations together and store on Drive:

```
/MyDrive/PCBSegClassNet/
    data_raw.zip                  ← contains: pcb_image/*.png  +  smd_annotation/*.csv
    checkpoints/                  ← (optional, for resume / saved best models)
```

To make the zip on a Windows host:

```powershell
Compress-Archive -Path data\pcb_image, data\smd_annotation -DestinationPath data_raw.zip -Force
```

Why unzip to local disk and not stream from Drive? Drive mounts thousands of small files extremely slowly (API throttling). Always unpack to `/content` for training.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Adjust path if you stored the raw zip elsewhere
RAW_ZIP = "/content/drive/MyDrive/PCBSegClassNet/data_raw.zip"

import os, time
assert os.path.exists(RAW_ZIP), f"Not found: {RAW_ZIP}"

%cd /content/PCBSegClassNet
!mkdir -p data
t0 = time.time()
!unzip -q -o {RAW_ZIP} -d data/
print(f"Unzip done in {time.time()-t0:.1f}s")

!echo "--- pcb_image:"; ls data/pcb_image/ | wc -l
!echo "--- smd_annotation:"; ls data/smd_annotation/ | wc -l

## 5. Generate masks + classification crops (`create_mask.py`)

Runs through all annotation CSVs, fills polygon masks per component class, and writes:
- `data/segmentation/images/` — HSI + CLAHE preprocessed PCB images
- `data/segmentation/masks/` — RGB masks (color-encoded per class)
- `data/classification/images/<CLASS>/` — individual component crops upscaled with the EDSR super-resolution model in `checkpoints/super_resolution.h5`

GPU-accelerated via the EDSR forward pass. Expect ~10–30 minutes depending on Colab GPU (A100 fastest).

In [ ]:
%cd /content/PCBSegClassNet/src/data
!python create_mask.py \
    -i ../../data/pcb_image/ \
    -a ../../data/smd_annotation/ \
    -id ../../data/segmentation/images \
    -ad ../../data/segmentation/masks \
    -cd ../../data/classification/images/

!echo "--- segmentation/images: $(ls ../../data/segmentation/images 2>/dev/null | wc -l)"
!echo "--- segmentation/masks:  $(ls ../../data/segmentation/masks  2>/dev/null | wc -l)"
!echo "--- classification crops total: $(find ../../data/classification/images -type f | wc -l)"
!echo "--- classification classes:"; ls ../../data/classification/images 2>/dev/null

## 6. Cut 768 px patches and split into train/val (`create_patches.py`)

Cuts the full PCB images + masks into 768×768 patches and moves the patches + classification crops into `train/` and `val/` subfolders (80/20 split). Pure CPU work, ~5 minutes.

After this cell, the dataset layout matches what the training scripts expect:

```
data/segmentation/train/{images,masks}/*.png
data/segmentation/val/{images,masks}/*.png
data/classification/train/<CLASS>/*.png
data/classification/val/<CLASS>/*.png
```

## 7. (Optional) Mirror checkpoints to Drive for persistence

Colab local disk is wiped on session end. Save best model files back to Drive at the end of training (or set up a callback). For now, just record the path.

## 5. (Optional) Mirror checkpoints to Drive for persistence

Colab local disk is wiped on session end. Save best model files back to Drive at the end of training (or set up a callback). For now, just record the path.

## 8. Train segmentation

Default config in `cfs/pscn_seg.yml` is `batch_size=16`, `epochs` controlled by `-epoch`.

**First run a 5-epoch sanity pass.** If loss is finite and val_dice_coef is improving, kick off the full 100 epochs.

## 6. Train segmentation

Default config in `cfs/pscn_seg.yml` is `batch_size=16`, `epochs` controlled by `-epoch`.

**First run a 5-epoch sanity pass.** If loss is finite and val_dice_coef is improving, kick off the full 100 epochs.

In [ ]:
# Full training run
%cd /content/PCBSegClassNet/src
!python train_segmentation.py -opt cfs/pscn_seg.yml -epoch 40

In [ ]:
# Full training run
%cd /content/PCBSegClassNet/src
!python train_segmentation.py -opt cfs/pscn_seg.yml -epoch 100

## 9. Train classification

## 7. Train classification

In [ ]:
%cd /content/PCBSegClassNet/src
!python train_classification.py -opt cfs/pscn_class.yml -epoch 40

In [ ]:
%cd /content/PCBSegClassNet/src
!python train_classification.py -opt cfs/pscn_class.yml -epoch 100

## 10. (Optional) Evaluate without retraining

Pass `-epoch 0` to skip training; the script will load `best_*.h5` from `checkpoints/` and run `model.evaluate(val_dataset)`. Make sure the checkpoint is in `/content/PCBSegClassNet/checkpoints/` (copy it back from Drive if you reconnected).

## 8. (Optional) Evaluate without retraining

Pass `-epoch 0` to skip training; the script will load `best_*.h5` from `checkpoints/` and run `model.evaluate(val_dataset)`. Make sure the checkpoint is in `/content/PCBSegClassNet/checkpoints/` (copy it back from Drive if you reconnected).

In [ ]:
# Restore checkpoints from Drive after a fresh session
!mkdir -p /content/PCBSegClassNet/checkpoints
!cp {DRIVE_CKPT_DIR}/best_seg.h5 /content/PCBSegClassNet/checkpoints/ 2>/dev/null || echo 'no seg ckpt'
!cp {DRIVE_CKPT_DIR}/best_class.h5 /content/PCBSegClassNet/checkpoints/ 2>/dev/null || echo 'no class ckpt'

%cd /content/PCBSegClassNet/src
!python train_segmentation.py -opt cfs/pscn_seg.yml -epoch 0